# VisTAD: 2×2 Student Architecture × Teacher Manifold Quality Cross-Ablation

**Experimental Hypothesis:**
Isolates the performance contributions of:
1. **Student Architecture Inductive Biases**: Domain-Coupled Subsystem TCNs vs. Scaled Permutation-Equivariant Universal Foundation (`UniversalFoundationTCN_Pro`).
2. **Teacher Manifold Supervision**: 2048-d Early Multimodal VLM features vs. 1024-d Ultra-VETime / DINOv2-Large foundation embeddings.

**Factorial Matrix:**
- **Cell (1,1)**: Old Student + Old Cache (2048-d)
- **Cell (1,2)**: Old Student + New Cache (1024-d)
- **Cell (2,1)**: New Student + Old Cache (2048-d)
- **Cell (2,2)**: New Student + New Cache (1024-d)


In [ ]:
# --- Step 1: Workspace Setup & Codebase Unpacking ---
import os, sys, shutil, subprocess, base64, io, zipfile

repo_dir = '/kaggle/working/CrossAblation'
os.makedirs(repo_dir, exist_ok=True)
os.chdir(repo_dir)

print('Unpacking self-contained Cross-Ablation codebase...')
b64_bundle = "..."
raw_zip = base64.b64decode(b64_bundle)
with zipfile.ZipFile(io.BytesIO(raw_zip)) as zf:
    zf.extractall('.')
print('Codebase unpacked successfully.')
print(f'Extracted top-level files: {sorted(os.listdir("."))}')


In [ ]:
# --- Step 2: GPU Accelerator Diagnostics ---
import torch
print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()
    print(f'Detected {n_gpus} GPU(s):')
    for i in range(n_gpus):
        prop = torch.cuda.get_device_properties(i)
        print(f'  [GPU {i}] {prop.name} | VRAM: {prop.total_memory / 1e9:.2f} GB')


In [ ]:
# --- Step 3: Multi-Dataset & Dual-Cache Auto-Discovery ---
import os, glob, shutil

for ds in ['swat', 'smd/test', 'smd/test_label', 'gecco', 'msl']:
    os.makedirs(os.path.join('data', ds), exist_ok=True)
os.makedirs('cache', exist_ok=True)
os.makedirs('checkpoints/cross_ablation', exist_ok=True)
os.makedirs('results/ablations', exist_ok=True)

# 1. SWaT
swat_files = glob.glob('/kaggle/input/**/SWaT_Dataset_*.csv', recursive=True) + glob.glob('/kaggle/input/**/swat/*.csv', recursive=True)
for p in set(swat_files):
    fname = os.path.basename(p)
    target = os.path.join('data/swat', fname)
    if not os.path.exists(target):
        try: os.symlink(p, target)
        except OSError: shutil.copy2(p, target)
print(f'Linked {len(os.listdir("data/swat"))} SWaT files.')

# 2. SMD
smd_test_files = glob.glob('/kaggle/input/**/smd/test/*.txt', recursive=True)
for p in smd_test_files:
    target = os.path.join('data/smd/test', os.path.basename(p))
    if not os.path.exists(target):
        try: os.symlink(p, target)
        except OSError: shutil.copy2(p, target)

smd_label_files = glob.glob('/kaggle/input/**/smd/test_label/*.txt', recursive=True)
for p in smd_label_files:
    target = os.path.join('data/smd/test_label', os.path.basename(p))
    if not os.path.exists(target):
        try: os.symlink(p, target)
        except OSError: shutil.copy2(p, target)
print(f'Linked {len(os.listdir("data/smd/test"))} SMD test files.')

# 3. GECCO
gecco_files = glob.glob('/kaggle/input/**/gecco_water.csv', recursive=True) + glob.glob('/kaggle/input/**/gecco/*.csv', recursive=True)
for p in set(gecco_files):
    target = os.path.join('data/gecco', os.path.basename(p))
    if not os.path.exists(target):
        try: os.symlink(p, target)
        except OSError: shutil.copy2(p, target)
print(f'Linked {len(os.listdir("data/gecco"))} GECCO files.')

# 4. MSL
msl_files = glob.glob('/kaggle/input/**/msl_*.csv', recursive=True) + glob.glob('/kaggle/input/**/msl/*.csv', recursive=True)
for p in set(msl_files):
    target = os.path.join('data/msl', os.path.basename(p))
    if not os.path.exists(target):
        try: os.symlink(p, target)
        except OSError: shutil.copy2(p, target)
print(f'Linked {len(os.listdir("data/msl"))} MSL files.')

# 5. Teacher Caches
cache_files = glob.glob('/kaggle/input/**/*.pt', recursive=True)
for p in set(cache_files):
    if 'cache' in p.lower() or 'ultravetime' in p.lower():
        target = os.path.join('cache', os.path.basename(p))
        if not os.path.exists(target):
            try: os.symlink(p, target)
            except OSError: shutil.copy2(p, target)
print(f'Discovered and linked {len(os.listdir("cache"))} caches in cache/: {sorted(os.listdir("cache"))}')


In [ ]:
# --- Step 4: Execute 6-Model Cross-Ablation Suite ---
import os, sys, subprocess, time
os.environ['PYTHONPATH'] = '.'

cmd = [
    sys.executable, 'run_cross_student_teacher_ablation.py',
    '--data_dir', 'data',
    '--cache_dir', 'cache',
    '--datasets', 'swat', 'gecco', 'smd', 'msl',
    '--epochs', '10',
    '--batch_size', '32',
    '--checkpoint_dir', 'checkpoints/cross_ablation',
    '--output_json', 'results/ablations/cross_student_teacher_ablation.json',
    '--output_report', 'results/ablations/CROSS_STUDENT_TEACHER_REPORT.md'
]

print(f'Executing command: {" ".join(cmd)}')
t0 = time.time()
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in iter(process.stdout.readline, ''):
    print(line, end='', flush=True)
process.stdout.close()
rc = process.wait()
print(f'Cross-Ablation process exited with return code: {rc} (Elapsed: {(time.time() - t0)/60:.2f} mins)')
if rc != 0:
    raise RuntimeError(f'Cross-Ablation failed with return code {rc}')


In [ ]:
# --- Step 5: Export Checkpoints & Reports to /kaggle/working/ ---
import os, shutil

out_ckpt = '/kaggle/working/checkpoints/cross_ablation'
out_res = '/kaggle/working/results/ablations'
os.makedirs(out_ckpt, exist_ok=True)
os.makedirs(out_res, exist_ok=True)

# Copy checkpoints
src_ck = 'checkpoints/cross_ablation'
if os.path.exists(src_ck):
    for f in os.listdir(src_ck):
        p = os.path.join(src_ck, f)
        shutil.copy2(p, os.path.join(out_ckpt, f))
        print(f'Exported: {f} ({os.path.getsize(p)/1e6:.2f} MB)')

# Copy results
src_re = 'results/ablations'
if os.path.exists(src_re):
    for f in os.listdir(src_re):
        p = os.path.join(src_re, f)
        shutil.copy2(p, os.path.join(out_res, f))
        print(f'Exported result artifact: {f}')

report_p = 'results/ablations/CROSS_STUDENT_TEACHER_REPORT.md'
if os.path.exists(report_p):
    print('\n' + '=' * 80)
    print('FINAL 2x2 FACTORIAL CROSS-ABLATION REPORT:')
    print('=' * 80)
    with open(report_p, 'r') as rf:
        print(rf.read())
